# Finetune a SmoLm model with 1.7 billion parameters using LoRa

We use a self-curated dataset with [arXiv](https://arxiv.org/) as a basis. The whole dataset is available from [Kaggle](https://www.kaggle.com/datasets/Cornell-University/arxiv). The dataset was filtered to have either `LLM` (only as a whole word), `Large Language Model`, `GRPO` or `GSPO` in the title of the abstract. 

We use the following system prompt:
```
You are an educated researcher and always answer in correct scientific terms.
You are very deep into LLMs and its methodologies. You are very creative.
```

The model should learn how to write abstracts about a certain topic. An example user prompt would be:
```
Write an abstract with the title 'Few-shot training LLMs for project-specific code-summarization'
```

The example abstract is taken from arXiv, so the answer will be:
```
Very large language models (LLMs), such as GPT-3 and Codex have achieved
state-of-the-art performance on several natural-language tasks, and show great
promise also for code. A particularly exciting aspect of LLMs is their knack
for few-shot and zero-shot learning: they can learn to perform a task with very
few examples. Few-shotting has particular synergies in software engineering,
where there are a lot of phenomena (identifier names, APIs, terminology, coding
patterns) that are known to be highly project-specific. However,
project-specific data can be quite limited, especially early in the history of
a project; thus the few-shot learning capacity of LLMs might be very relevant.
In this paper, we investigate the use few-shot training with the very large GPT
(Generative Pre-trained Transformer) Codex model, and find evidence suggesting
that one can significantly surpass state-of-the-art models for
code-summarization, leveraging project-specific training.
```

If finetuning works, we will observe that the models know about `GRPO` which
was introduced *after* the knowledge cutoff. Moreover, the models should adhere
to the format and scientific language of the abstract.

Implementation ideas and parts of the script from https://www.philschmid.de/fine-tune-llms-in-2025
Implementation ideas and script from https://www.philschmid.de/fine-tune-llms-in-2025

In [1]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, ModelConfig, SFTConfig, get_peft_config
from datasets import load_dataset
# much faster kernel
from liger_kernel.transformers import AutoLigerKernelForCausalLM

# Arguments and Configuration

In [2]:
model_name = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
dataset_name = 'llm-abstract-dataset.jsonl.xz'
output_dir = "runs/" + model_name.split("/")[-1] + "-" + dataset_name.split(".")[0]

In [3]:
model_args = ModelConfig(model_name_or_path=model_name, 
                         model_revision='main', 
                         dtype='bfloat16', 
                         use_peft=True, 
                         lora_r=16, 
                         lora_alpha=16, 
                         lora_dropout=0.05, 
                         lora_target_modules='all-linear', 
                         lora_modules_to_save=['lm_head', 'embed_tokens'],
                         lora_task_type='CAUSAL_LM', 
                         use_rslora=False, 
                        )

In [4]:
training_args = SFTConfig(
     output_dir=output_dir,    
     num_train_epochs=4,
     bf16=True,
     packing=True,
     max_length=1024,
     per_device_train_batch_size=8,
     gradient_accumulation_steps=2,
     gradient_checkpointing=True,
     gradient_checkpointing_kwargs = { "use_reentrant": False },
     learning_rate=2.0e-4,
     lr_scheduler_type="constant",
     use_liger_kernel=True,
     warmup_steps=100,
)

# Load dataset

In [5]:
train_dataset = load_dataset("json", data_files=dataset_name, split="train")

f'Dataset with {len(train_dataset)} samples and the following features: {train_dataset.features}'

"Dataset with 16435 samples and the following features: {'messages': List({'role': Value('string'), 'content': Value('string')})}"

# Tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token

# Model

In [7]:
# define model kwargs
model_kwargs = dict(
    revision=model_args.model_revision, # What revision from Huggingface to use, defaults to main
    dtype=model_args.dtype, # What torch dtype to use, defaults to auto
    use_cache=False if training_args.gradient_checkpointing else True, # Whether
    low_cpu_mem_usage=True,  # Reduces memory usage on CPU for loading the model
)

In [8]:
peft_config = get_peft_config(model_args)

In [9]:
model = AutoLigerKernelForCausalLM.from_pretrained(model_args.model_name_or_path, **model_kwargs)

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

# Trainer

In [10]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

/home/cwinkler/uv/finetuning-env/.venv/lib/python3.13/site-packages/peft/tuners/tuners_utils.py:1463: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


trainable params: 219,414,528 || all params: 1,930,790,912 || trainable%: 11.3640


/tmp/ipykernel_1027628/2615974910.py:1: FutureWarning: `use_liger_kernel=True` is deprecated and will be removed in v2.0.0. Use the Hub kernels instead, with `model_init_kwargs={"use_kernels": True}`.
  trainer = SFTTrainer(


## Training loop

In [11]:
train_result = trainer.train()
# log metrics
metrics = train_result.metrics
metrics['train_samples'] = len(train_dataset)
trainer.save_metrics('train', metrics)
trainer.save_state()
metrics

/home/cwinkler/uv/finetuning-env/.venv/lib/python3.13/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)
Backend impl 'liger_kernel.ops.backends._ascend.rope' for op 'rope' could not be imported: ImportError: cannot import name 'tanh' from 'triton.language.math' (/home/cwinkler/uv/finetuning-env/.venv/lib/python3.13/site-packages/triton/language/math.py)


Step,Training Loss
10,2.662724
20,2.226385
30,2.120377
40,2.067946
50,2.051182
60,2.011064
70,2.029713
80,2.001541
90,2.001884
100,2.002481


{'train_runtime': 4184.6204,
 'train_samples_per_second': 5.356,
 'train_steps_per_second': 0.336,
 'total_flos': 2.2640731483840512e+17,
 'train_loss': 1.829382692307149,
 'entropy': 1.7172479799815588,
 'mean_token_accuracy': 0.6080670016152518,
 'num_tokens': 21818636.0,
 'epoch': 4.0,
 'train_samples': 16435}

In [ ]:
!nvidia-smi

# Save model

In [ ]:
# Restore k,v cache for fast inference
trainer.model.config.use_cache = True
trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)
f"saved to {training_args.output_dir}"